# 3-way conversation advanced exercises

A 3-way coversation between GPT, Gemini and local running Ollama about the results of a motorsport race.

In [ ]:
import os
import requests
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

# Path(__file__) no funciona bien en Notebooks, usamos Path().cwd()
# .parent sube a 'week2', el segundo .parent sube a la RAÍZ
root_path = Path.cwd().parent.parent
env_path = root_path / ".env"

if env_path.exists():
    load_dotenv(dotenv_path=env_path)
    print("✅ File .env load success")
else:
    print(f"❌ No .env found in: {env_path}")

In [ ]:
# ---------- Setup clients ----------

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")

openai = OpenAI(api_key=openai_api_key)

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)


GPT_MODEL = "gpt-4.1-mini"
GEMINI_MODEL = "gemini-2.5-flash"       # or "gemini-2.5-flash-lite"
LLAMA_MODEL = "llama3.2"                # pulled via: ollama pull llama3.2

In [ ]:
# ---------- Character system prompts ----------

ALPHONSE_SYSTEM = """
You are Alphonse, a Formula 1 fan who absolutely adores Max Verstappen comeback from the bottom up to the sixth position
by the end of the race.
You are stubborn, narrow-minded about other perspectives, and you instinctively argue against anything Bella says,
often without strong evidence. You are opinionated, defensive about Red Bull and Verstappen,
and you like to dismiss concerns about fairness or bias in F1.
Stay snarky, combative, and contrarian, but keep your language clean and non-abusive.
Always speak in the first person as Alphonse.
"""

BELLA_SYSTEM = """
You are Bella, a passionate Formula 1 enthusiast with deep, up-to-date knowledge of FIA rules and regulations.
You pay a lot of attention to fairness and equality among drivers, and you are particularly sensitive to how
Spanish-speaking drivers (like Franco Colapinto, Fernando Alonso, Carlos Sainz, Sergio Pérez, etc.) are treated. 
You noticed Colapinto being harshly punished for a mechanic touching the car before the race start while 
Bearman, Antonelli and even Russell moved their cars just before the red lights out but got no penalty.
You bring historical examples into the conversation.
Stay calm but firm, and back up your points with FIA rule awareness and race examples when possible.
Always speak in the first person as Bella.
"""

CORA_SYSTEM = """
You are Cora, the youngest and most easy-going of the three.
You are very curious, love asking questions, and you hate when the discussion becomes heated.
Whenever the tone escalates, you try to calm things down, reframe the topic more constructively,
and ask naive questions about Formula 1 fairness, or racing strategy.
You do NOT take strong sides; instead, you seek understanding and harmony.
Always speak in the first person as Cora, asking 1–3 questions in most replies.
"""

In [ ]:
# ---------- Helper to format conversation history ----------

def render_conversation(conversation):
    """
    conversation: list of dicts like {"speaker": "Bella", "text": "..."}
    Returns a readable transcript for prompts.
    """
    lines = []
    for turn in conversation:
        lines.append(f'{turn["speaker"]}: {turn["text"]}')
    return "\n".join(lines) if lines else "(No messages yet.)"

In [ ]:
# ---------- Model stream function ----------

def stream_chat(client, model, messages, speaker_name):
    """
    Streams tokens from an OpenAI-compatible chat endpoint,
    updating a Markdown cell live and returning the full text.
    """
    full_text = ""
    # Initial empty display for this speaker
    handle = display(Markdown(f"**{speaker_name}:** "), display_id=True)
    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True,
    )
    for chunk in stream:
        delta = chunk.choices[0].delta
        if not delta or not delta.content:
            continue
        text = delta.content
        full_text += text
        # Update the Markdown cell as text arrives
        handle.update(Markdown(f"**{speaker_name}:** {full_text}"))
    return full_text.strip()

In [ ]:
#---------- Model call functions ----------


def call_alphonse(conversation):
    transcript = render_conversation(conversation)
    user_prompt = f"""
You are Alphonse in a 3-way group chat with Bella and Cora.

The topic is the 2026 Australia Formula 1 race from this weekend.
Here is the conversation so far (most recent messages at the bottom):

{transcript}

Now respond with what Alphonse would say next.
- Be a stubborn Max Verstappen fan.
- Instinctively argue against Bella's points, even with weak reasoning.
- Keep it snarky and combative but not abusive.
- Stay focused on Formula 1 and the recent race.
Write 2–4 sentences as Alphonse only.
"""
    messages = [
        {"role": "system", "content": ALPHONSE_SYSTEM},
        {"role": "user", "content": user_prompt},
    ]
    return stream_chat(openai, GPT_MODEL, messages, "Alphonse")

def call_bella(conversation):
    transcript = render_conversation(conversation)
    user_prompt = f"""
You are Bella in a 3-way group chat with Alphonse and Cora.

The topic is the 2026 Australia Formula 1 race from this weekend.
Here is the conversation so far (most recent messages at the bottom):

{transcript}

Now respond with what Bella would say next.
- Bring in your deep knowledge of F1 and FIA rules.
- Highlight fairness and equality among drivers, especially Spanish-speaking drivers,
  and how you feel they are not being respected.
- Use concrete examples when you can (even if approximate).
- Stay calm but firm, and respond to any dismissals from Alphonse.
Write 2–4 sentences as Bella only.
"""
    messages = [
        {"role": "system", "content": BELLA_SYSTEM},
        {"role": "user", "content": user_prompt},
    ]
    return stream_chat(gemini, GEMINI_MODEL, messages, "Bella")

def call_cora(conversation):
    transcript = render_conversation(conversation)
    user_prompt = f"""
You are Cora in a 3-way group chat with Alphonse and Bella.

The topic is the 2026 Australia Formula 1 race from this weekend.
Here is the conversation so far (most recent messages at the bottom):

{transcript}

Now respond with what Cora would say next.
- Notice if Alphonse and Bella are arguing or the tone is heated, and actively try to calm things down.
- Be curious and ask multiple sincere questions about Formula 1, fairness, strategy, or the race.
- Do not take strong sides; instead, encourage understanding and more balanced discussion.
Write 2–4 sentences as Cora only, and include 1–3 questions.
"""
    messages = [
        {"role": "system", "content": CORA_SYSTEM},
        {"role": "user", "content": user_prompt},
    ]
    return stream_chat(ollama, LLAMA_MODEL, messages, "Cora")

In [ ]:
# ---------- Run 6 rounds of conversation ----------

def run_conversation(rounds: int = 6):
    conversation = []

    # Optional: let Bella start, since she's the F1 expert
    print("=== Initial message from Bella ===")
    first_bella = call_bella(conversation)
    conversation.append({"speaker": "Bella", "text": first_bella})
    #print(f"Bella: {first_bella}\n")

    for r in range(1, rounds + 1):
        print(f"=== Round {r} ===")

        # Alphonse responds
        alphonse_msg = call_alphonse(conversation)
        conversation.append({"speaker": "Alphonse", "text": alphonse_msg})
        #print(f"Alphonse: {alphonse_msg}\n")

        # Bella responds
        bella_msg = call_bella(conversation)
        conversation.append({"speaker": "Bella", "text": bella_msg})
        #print(f"Bella: {bella_msg}\n")

        # Cora responds
        cora_msg = call_cora(conversation)
        conversation.append({"speaker": "Cora", "text": cora_msg})
        #print(f"Cora: {cora_msg}\n")

    return conversation

if __name__ == "__main__":
    run_conversation(rounds=6)